# CoatShield boundary finder: train on Kaggle

Pretrains the compact U-Net on OCT5k (real retinal OCT), fine-tunes it on synthetic pellet
scans, trains the same network from scratch for comparison, evaluates on the synthetic
validation set, exports ONNX (opset 17) and writes the manifest.

Before running: turn the GPU on (Settings > Accelerator), and attach two private Kaggle
datasets: `oct5k` (the `data/raw/oct5k/OCT5k` folder) and `coatshield-synthetic`
(the `data/synthetic` folder with `train/` and `val/`). The locked test set is never uploaded.
Set `REPO_URL` to the repository.

In [ ]:
REPO_URL = "https://github.com/<owner>/coatshield.git"  # set this
OCT5K_INPUT = "/kaggle/input/oct5k/OCT5k"
SYNTHETIC_INPUT = "/kaggle/input/coatshield-synthetic"

import os, subprocess
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/coatshield"], check=True)
os.chdir("/kaggle/working/coatshield")
# torch and torchvision are preinstalled on Kaggle; install the rest.
subprocess.run("grep -v -E '^(torch|torchvision|--extra-index-url)' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt", shell=True, check=True)
os.makedirs("data/raw/oct5k", exist_ok=True)
for link, target in (("data/raw/oct5k/OCT5k", OCT5K_INPUT), ("data/synthetic/train", SYNTHETIC_INPUT + "/train"), ("data/synthetic/val", SYNTHETIC_INPUT + "/val")):
    if os.path.lexists(link):
        os.remove(link) if os.path.islink(link) else None
    if not os.path.exists(link):
        os.symlink(target, link)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - turn the accelerator on")

## 1. Pretrain on OCT5k (split by patient volume)

In [ ]:
import json
from coatshield.config import load_config
from coatshield.seg.train import run_stage

cfg = load_config()
device = "cuda" if torch.cuda.is_available() else "cpu"
pre = run_stage("pretrain", cfg, device, workers=2)
print(json.dumps({k: v for k, v in pre.items() if k != "history"}, indent=1))

## 2. Fine-tune on synthetic pellets, and 3. the same network from scratch

In [ ]:
fine = run_stage("finetune", cfg, device, workers=2)
scratch = run_stage("scratch", cfg, device, workers=2)
for name, info in (("fine-tuned", fine), ("from scratch", scratch)):
    print(name, info["val_boundaries"], "parameters", info["parameters"])

## 4. Evaluate, export ONNX and write the manifest

In [ ]:
subprocess.run(["python", "scripts/evaluate_seg.py"], check=True)
subprocess.run(["python", "scripts/export_onnx.py", "--checkpoint", "unet_pellets", "--name", "unet"], check=True)
print(open("models/manifest.json").read())

## 5. Collect the outputs

Download `coatshield_models.zip` and unpack it at the repository root: it holds `models/unet.onnx`, `models/manifest.json`, the checkpoints and the evaluation report.

In [ ]:
subprocess.run("zip -q -r /kaggle/working/coatshield_models.zip models reports/seg_*", shell=True, check=True)
print(os.path.getsize("/kaggle/working/coatshield_models.zip") / 1e6, "MB")